# Week 2 Project 1 — Local Ollama Gradio Chatbot

This notebook adapts the Week 2 lessons on multi-model access, Ollama's OpenAI-compatible API, Gradio interfaces, streaming responses, and conversation history. It runs locally with Ollama and does not require an API key.

Set `RUN_UI = True` only after the cells above it work. The UI stays local by default.

## Learning goals

- Call a local Ollama model through its HTTP chat API.
- Keep system, user, and assistant messages in a chat history.
- Stream a response into a simple Gradio chatbot.
- Compare installed local models without coupling the app to a vendor SDK.
- Separate the model adapter from the user interface.

In [ ]:
import json
import time
from typing import Any

import requests
from IPython.display import Markdown, display

OLLAMA_URL = "http://127.0.0.1:11434"
MODEL = "llama3.2:1b"  # Change to any model returned by /api/tags.
REQUEST_TIMEOUT = 180
SYSTEM_PROMPT = "You are a concise, helpful assistant. Explain your reasoning briefly and clearly."
RUN_UI = False

In [ ]:
def ollama_tags() -> list[str]:
    """Return installed model names from the local Ollama server."""
    response = requests.get(f"{OLLAMA_URL}/api/tags", timeout=15)
    response.raise_for_status()
    return [item["name"] for item in response.json().get("models", [])]

try:
    installed = ollama_tags()
    print("Ollama is reachable.")
    print("Installed models:", installed or "none")
except requests.RequestException as exc:
    installed = []
    print("Ollama is not reachable yet:", exc)
    print("Start Ollama and run: ollama pull llama3.2:1b")

In [ ]:
def chat_once(messages: list[dict[str, str]], model: str = MODEL, temperature: float = 0.2) -> str:
    """Call Ollama's native /api/chat endpoint and return one complete answer."""
    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {"temperature": temperature, "num_ctx": 4096},
    }
    response = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    return response.json()["message"]["content"].strip()


def chat_stream(messages: list[dict[str, str]], model: str = MODEL, temperature: float = 0.2) -> str:
    """Stream tokens from Ollama and return the complete answer."""
    payload = {
        "model": model,
        "messages": messages,
        "stream": True,
        "options": {"temperature": temperature, "num_ctx": 4096},
    }
    response = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, stream=True, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    parts: list[str] = []
    for line in response.iter_lines(decode_unicode=True):
        if not line:
            continue
        event = json.loads(line)
        token = event.get("message", {}).get("content", "")
        if token:
            print(token, end="", flush=True)
            parts.append(token)
        if event.get("done"):
            break
    print()
    return "".join(parts).strip()

In [ ]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Explain why a chat application should keep conversation history."},
]
if installed:
    answer = chat_stream(messages)
    display(Markdown(answer))
else:
    print("Install a model and rerun this cell.")

## Multi-model comparison

If several models are installed, the same adapter can compare them. The function falls back to the configured model when only one model is present.

In [ ]:
def compare_models(prompt: str, models: list[str] | None = None) -> dict[str, str]:
    choices = models or installed or [MODEL]
    results = {}
    for name in choices[:3]:
        try:
            results[name] = chat_once([
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ], model=name)
        except requests.RequestException as exc:
            results[name] = f"Request failed: {exc}"
    return results

if installed:
    for name, text in compare_models("Give one practical design rule for a reliable chatbot.").items():
        display(Markdown(f"### {name}\n{text}"))
else:
    print("Comparison will run after Ollama has at least one installed model.")

## Gradio chat interface

This uses Gradio's `ChatInterface` and the same Ollama adapter. It is intentionally disabled during **Run All** so a notebook run finishes. Set `RUN_UI = True` and rerun this cell to open the local UI.

In [ ]:
try:
    import gradio as gr
except ImportError:
    gr = None
    print("Install the UI dependency with: python -m pip install 'gradio>=5,<6'")


def gradio_reply(message: str, history: list[dict[str, str]], selected_model: str, system_prompt: str) -> str:
    messages = [{"role": "system", "content": system_prompt}]
    for item in history or []:
        role = item.get("role")
        content = item.get("content", "")
        if role in {"user", "assistant"} and isinstance(content, str):
            messages.append({"role": role, "content": content})
    messages.append({"role": "user", "content": message})
    return chat_once(messages, model=selected_model or MODEL)

if gr is not None:
    model_choices = installed or [MODEL]
    demo = gr.ChatInterface(
        fn=gradio_reply,
        additional_inputs=[
            gr.Dropdown(model_choices, value=model_choices[0], label="Ollama model"),
            gr.Textbox(value=SYSTEM_PROMPT, label="System prompt"),
        ],
        title="Week 2 — Local Ollama Chatbot",
        description="A local Gradio chat UI backed by Ollama.",
    )
    if RUN_UI:
        demo.launch(inbrowser=False, share=False)
    else:
        print("UI built. Set RUN_UI = True and rerun this cell to launch it.")

## Exercises

1. Install a second Ollama model and compare its answer with `compare_models`.
2. Add a response-length instruction to `SYSTEM_PROMPT` and observe the effect.
3. Add a reset button to the Gradio app.
4. Replace `chat_once` with a streaming generator and wire it to a custom `gr.Blocks` chat UI.